In [3]:
# Step 1: Install scispacy and the model
#!pip install scispacy
#!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Step 2: Import required libraries
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
import json

In [20]:
#print(info["studies"])

In [22]:
'''Sample of pulling only 10 studies related information'''
# API endpoint for Major Depressive Disorder
#url = "https://clinicaltrials.gov/api/v2/studies?query.cond='Major%20Depressive%20Disorder'"
new_url = "https://clinicaltrials.gov/api/v2/studies?query.term=Drug&query.cond=Major%20Depressive%20Disorder"

# Make the GET request
response = requests.get(new_url)

# Check if request was successful
if response.status_code == 200:
    #json_data = response.json()
    json_text = response.text

    info = json.loads(json_text)
    
j = 0 
df = pd.DataFrame(columns=["nct_id", "Study_title", "Study_status","Study_start_date"
                           ,"Study_lead_sponsor_or_company","Indications/Conditions","Drug_names"
                          ,"Primary_outcomes","Secondary_outcome","Eligibilty_criteria"
                          ,"Primary Completion Date","Completion Date","First Posted"
                          ,"Results First Posted","Last Update Posted"])
for i in info["studies"]:
    #if i['protocolSection']['armsInterventionsModule']['interventions']['type'] == 'DRUG':        
    j = j + 1
    print("-------------%d study--------------"%j)
    #print(i)
    nct_id = i['protocolSection']['identificationModule']['nctId']
    official_title = i['protocolSection']['identificationModule']['officialTitle']
    overall_status = i['protocolSection']['statusModule']['overallStatus']
    study_start_date = i['protocolSection']['statusModule']['startDateStruct']['date']
    lead_sponsor = i['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']['name']
    
    #conditions_list = (i['protocolSection']['conditionsModule']['conditions'])
    #conditions = "; ".join(conditions_list)
    #print(i['protocolSection']['armsInterventionsModule'])
    print("NCT ID: %s" %nct_id)
    print("Study title: %s" %official_title)
    print("Study Status: %s" %overall_status)
    print("Study start date: %s" %study_start_date)
    print("Study Lead Sponsor/Company: %s" %lead_sponsor)
    #print(conditions)
    
    #interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", {})
    conditions = []
    for x in i['derivedSection']['conditionBrowseModule']['meshes']: 
        #print(x['term'])
        if x['term'].lower not in conditions: 
            conditions.append(x['term'])
        else: 
            continue
    print("Indications/Conditions treated :%s" %conditions)
    #for x in i['protocolSection']:
    #    print(type(x))
    drug_names = []
    try: 
        value_parent =  i['protocolSection']['armsInterventionsModule']['interventions']
        #print(value_parent)
        for z in value_parent: 
            #print(type(i))
            if z['type']=='DRUG':
                value = z['name']
                drug_names.append(value)
            else:
                continue
    except: 
        print('No interventions found for %s' %nct_id)
    print("Drug names :%s" %drug_names)
    #print(i['protocolSection']['outcomesModule'])
    
    try: 
        value_parent_1 = i['protocolSection']['outcomesModule']['primaryOutcomes']
        for a_2 in value_parent_1: 
            #print(type(a_2))
            primary_outcomes = (";;; ".join(list(a_2.values())))    
    except: 
        continue
    print("Primary Outcomes: %s" %primary_outcomes)
    
    try: 
        value_parent_2 = i['protocolSection']['outcomesModule']['secondaryOutcomes']
        for a_3 in value_parent_2: 
            #print(type(a_2))
            secondary_outcomes = (";;; ".join(list(a_3.values())))    
    except: 
        continue
    print("Secondary Outcomes: %s" %secondary_outcomes)

    try: 
        value_parent_3 = i['protocolSection']['eligibilityModule']
        #print(value_parent_3)
        eligibility = ";;; ".join(f"{key}: {value}" for key, value in value_parent_3.items())
        print("Eligibilty Criteria: %s" %eligibility)
        #for a_2 in value_parent_1: 
            #print(type(a_2))
        #    primary_outcomes = (";;; ".join(list(a_2.values())))    
    except: 
        continue
    #print("Primary Outcomes: %s" %primary_outcomes)

    print('----------------------------------')

    #print(i["protocolSection"]["statusModule"]["primaryCompletionDateStruct"]["date"])
    # Primary Completion Date
    try: 
        primary_completion_date = i["protocolSection"]["statusModule"]["primaryCompletionDateStruct"]["date"]
    except: 
        continue

    # Completion Date
    try: 
        completion_date = i["protocolSection"]["statusModule"]["completionDateStruct"]["date"]
    except: 
        continue

    # Study first posted Date
    try: 
        study_first_posted_date = i["protocolSection"]["statusModule"]["studyFirstPostDateStruct"]["date"]
    except: 
        continue

    # results first posted Date
    try: 
        results_first_posted_date = i["protocolSection"]["statusModule"]["resultsFirstPostDateStruct"]["date"]
    except: 
        continue

    # Last update post Date
    try: 
        last_update_post_date = i["protocolSection"]["statusModule"]["lastUpdatePostDateStruct"]["date"]
    except: 
        continue

    # Results Overview (participant flow module)
    try:
        results_overview_raw = i["resultsSection"]["participantFlowModule"]
        results_overview = json.dumps(results_overview_raw)
    except:
        results_overview = None
    
    # Baseline Characteristics
    try:
        baseline_chars_raw = i["resultsSection"]["baselineCharacteristicsModule"]
        baseline_chars = json.dumps(baseline_chars_raw)
    except:
        baseline_chars = None

    
    #print(x)
    #print(interventions_module)
    #for i in interventions_module: 
    #    print('-----')
    #    print(i)
    #else: 
    #    continue

    df.loc[len(df)] = [nct_id,official_title,overall_status,study_start_date,lead_sponsor,conditions
                       ,drug_names,primary_outcomes,secondary_outcomes,eligibility,primary_completion_date
                      ,completion_date,study_first_posted_date,results_first_posted_date,last_update_post_date]

-------------1 study--------------
NCT ID: NCT00143091
Study title: A Six-Week, Fixed Dose, Double-Blind, Double-Dummy, Placebo and Sertraline Controlled, Multicentre Trial to Evaluate the Safety and Efficacy of CP-316,311 in Outpatients With Major Depressive Disorder
Study Status: TERMINATED
Study start date: 2005-04
Study Lead Sponsor/Company: Pfizer
Indications/Conditions treated :['Depressive Disorder', 'Depression', 'Depressive Disorder, Major']
Drug names :['CP-316,311', 'Placebo', 'Sertraline']
Primary Outcomes: Change from baseline in HAM-D (17) at the week 6 visit
Secondary Outcomes: Change from baseline in MADRS, HAM-A, CGI-S and CGI-I at the week 6 visit.
Eligibilty Criteria: eligibilityCriteria: Inclusion Criteria:

* Outpatients with DSM-IV major depressive disorder

Exclusion Criteria:

* Women of child bearing potential;;; healthyVolunteers: False;;; sex: ALL;;; minimumAge: 18 Years;;; stdAges: ['ADULT', 'OLDER_ADULT']
----------------------------------
-------------2 st

In [23]:
df

,nct_id,Study_title,Study_status,Study_start_date,Study_lead_sponsor_or_company,Indications/Conditions,Drug_names,Primary_outcomes,Secondary_outcome,Eligibilty_criteria,Primary Completion Date,Completion Date,First Posted,Results First Posted,Last Update Posted
0,NCT05165394,"A Randomized, Double-Blind, Placebo-Controlled...",COMPLETED,2021-11-30,Neurocrine Biosciences,"[Anhedonia, Depressive Disorder, Depression, D...","[Placebo, NBI-1065846]","Change in Anhedonia Severity, as Measured by C...","CGI-S Scores;;; The CGI-S scale, which is base...",eligibilityCriteria: Key Inclusion Criteria:\n...,2023-07-07,2023-09-13,2021-12-21,2024-07-31,2024-07-31
1,NCT00866294,"A Randomized, Double-blind, Placebo Controlled...",COMPLETED,2009-04,GlaxoSmithKline,"[Depressive Disorder, Depression, Depressive D...","[paroxetine IR 10mg tablet, paroxetine IR 20mg...",Adjusted Mean Change From Baseline in the Hami...,Percentage of Responders Based on the Clinical...,eligibilityCriteria: Inclusion Criteria:\n\n\<...,2010-02,2010-02,2009-03-20,2010-11-15,2017-01-30
2,NCT01784991,"""Does a 1 Plus 1 Hydromorphone Pain Protocol H...",TERMINATED,2011-12,Albert Einstein Healthcare Network,"[Respiratory Insufficiency, Depression, Depres...","[Hydromorphone, Usual care group]",Respiratory Depression;;; Incidence of respira...,Serious Adverse Events;;; Defined as prolonged...,eligibilityCriteria: Inclusion Criteria:\n\n1....,2014-04-16,2014-04-16,2013-02-06,2020-03-19,2020-03-19


In [41]:
import requests
import pandas as pd
import time

# Initialize empty DataFrame
df_1 = pd.DataFrame(columns=[
    "nct_id", "Study_title", "Study_status", "Study_start_date",
    "Study_lead_sponsor_or_company", "Indications/Conditions", "Drug_names",
    "Primary_outcomes", "Secondary_outcome", "Eligibilty_criteria",
    "Primary Completion Date", "Completion Date", "First Posted",
    "Results First Posted", "Last Update Posted", "Results_Overview",
    "Baseline_Characteristics"
])

# API setup
base_url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.term": "Drug",
    "query.cond": "Major Depressive Disorder",
    "pageSize": 100
}

next_page_token = None

# Paginate through all pages
while True:
    if next_page_token:
        params["pageToken"] = next_page_token

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Request failed: {e}")
        break

    for study in data.get("studies", []):
        try:
            nct_id = study['protocolSection']['identificationModule']['nctId']
            official_title = study['protocolSection']['identificationModule'].get('officialTitle', "")
            overall_status = study['protocolSection']['statusModule'].get('overallStatus', "")
            study_start_date = study['protocolSection']['statusModule'].get('startDateStruct', {}).get('date', "")
            lead_sponsor = study['protocolSection']['sponsorCollaboratorsModule']['leadSponsor'].get('name', "")

            # Conditions
            conditions = [x['term'] for x in study['derivedSection'].get('conditionBrowseModule', {}).get('meshes', [])]

            # Drugs
            drug_names = [z['name'] for z in study['protocolSection'].get('armsInterventionsModule', {}).get('interventions', []) if z.get('type') == 'DRUG']

            # Outcomes
            primary_outcomes = ";;; ".join(" ".join(str(v) for v in o.values()) for o in study['protocolSection'].get('outcomesModule', {}).get('primaryOutcomes', []))
            secondary_outcomes = ";;; ".join(" ".join(str(v) for v in o.values()) for o in study['protocolSection'].get('outcomesModule', {}).get('secondaryOutcomes', []))

            # Eligibility
            eligibility = ";;; ".join(f"{k}: {v}" for k, v in study['protocolSection'].get('eligibilityModule', {}).items())

            # Dates
            primary_completion_date = study['protocolSection']['statusModule'].get('primaryCompletionDateStruct', {}).get('date', "")
            completion_date = study['protocolSection']['statusModule'].get('completionDateStruct', {}).get('date', "")
            study_first_posted_date = study['protocolSection']['statusModule'].get('studyFirstPostDateStruct', {}).get('date', "")
            results_first_posted_date = study['protocolSection']['statusModule'].get('resultsFirstPostDateStruct', {}).get('date', "")
            last_update_post_date = study['protocolSection']['statusModule'].get('lastUpdatePostDateStruct', {}).get('date', "")

            # Add to DataFrame
            df_1.loc[len(df_1)] = [
                nct_id, official_title, overall_status, study_start_date, lead_sponsor,
                conditions, drug_names, primary_outcomes, secondary_outcomes, eligibility,
                primary_completion_date, completion_date, study_first_posted_date,
                results_first_posted_date, last_update_post_date
            ]
        except Exception as e:
            print(f"Error parsing study {study.get('nctId', 'Unknown')}: {e}")
            continue

    next_page_token = data.get("nextPageToken")
    if not next_page_token:
        break

    time.sleep(0.3)  # polite delay to avoid overloading the server

# Final DataFrame ready
print(f"Total studies fetched: {len(df)}")
#print(df_1.head())


Total studies fetched: 2196
        nct_id                                        Study_title  \
0  NCT00143091  A Six-Week, Fixed Dose, Double-Blind, Double-D...   
1  NCT00048594  A Double-Blind, Placebo-Controlled, Multicente...   
2  NCT05165394  A Randomized, Double-Blind, Placebo-Controlled...   
3  NCT06228391  Examining the Efficacy and Safety of Subanesth...   
4  NCT00866294  A Randomized, Double-blind, Placebo Controlled...   

         Study_status Study_start_date  \
0          TERMINATED          2005-04   
1           COMPLETED       2002-01-03   
2           COMPLETED       2021-11-30   
3  NOT_YET_RECRUITING          2024-03   
4           COMPLETED          2009-04   

                 Study_lead_sponsor_or_company  \
0                                       Pfizer   
1                      Merck Sharp & Dohme LLC   
2                       Neurocrine Biosciences   
3  Minneapolis Veterans Affairs Medical Center   
4                              GlaxoSmithKline   

   

In [43]:
len(df_1)

2196

In [44]:
df_1.head(5)

,nct_id,Study_title,Study_status,Study_start_date,Study_lead_sponsor_or_company,Indications/Conditions,Drug_names,Primary_outcomes,Secondary_outcome,Eligibilty_criteria,Primary Completion Date,Completion Date,First Posted,Results First Posted,Last Update Posted
0,NCT00143091,"A Six-Week, Fixed Dose, Double-Blind, Double-D...",TERMINATED,2005-04,Pfizer,"[Depressive Disorder, Depression, Depressive D...","[CP-316,311, Placebo, Sertraline]",Change from baseline in HAM-D (17) at the week...,"Change from baseline in MADRS, HAM-A, CGI-S an...",eligibilityCriteria: Inclusion Criteria:\n\n* ...,,2006-04,2005-09-02,,2008-03-24
1,NCT00048594,"A Double-Blind, Placebo-Controlled, Multicente...",COMPLETED,2002-01-03,Merck Sharp & Dohme LLC,"[Depressive Disorder, Depression, Depressive D...","[aprepitant, Comparator: placebo (unspecified)]",Total HAMD-17 score </= 10 after 10 weeks of t...,CGI-S score change from Week 10 at Month 6; HA...,eligibilityCriteria: Patients with Major Depre...,2003-12-18,2003-12-22,2002-11-05,,2017-05-05
2,NCT05165394,"A Randomized, Double-Blind, Placebo-Controlled...",COMPLETED,2021-11-30,Neurocrine Biosciences,"[Anhedonia, Depressive Disorder, Depression, D...","[Placebo, NBI-1065846]","Change in Anhedonia Severity, as Measured by C...",Change in Total MADRS Score From Baseline to D...,eligibilityCriteria: Key Inclusion Criteria:\n...,2023-07-07,2023-09-13,2021-12-21,2024-07-31,2024-07-31
3,NCT06228391,Examining the Efficacy and Safety of Subanesth...,NOT_YET_RECRUITING,2024-03,Minneapolis Veterans Affairs Medical Center,"[Brain Injuries, Brain Injuries, Traumatic, De...","[IV ketamine, Midazolam]",Change in Severity of Depressive Symptoms Tota...,Change in Severity of Post-traumatic Stress Di...,eligibilityCriteria: Inclusion Criteria:\n\n* ...,2026-12,2027-03,2024-01-29,,2024-01-29
4,NCT00866294,"A Randomized, Double-blind, Placebo Controlled...",COMPLETED,2009-04,GlaxoSmithKline,"[Depressive Disorder, Depression, Depressive D...","[paroxetine IR 10mg tablet, paroxetine IR 20mg...",Adjusted Mean Change From Baseline in the Hami...,Mean Change From Baseline in the HAM-D Total S...,eligibilityCriteria: Inclusion Criteria:\n\n\<...,2010-02,2010-02,2009-03-20,2010-11-15,2017-01-30


In [45]:
df_1.to_csv('Clinical_trails_V3.csv')